### Importeer de dataset

We gaan nu de sklearn library voor het neurale netwerk gebruiken om te runnen op onze dataset. 

Je kunt in de huiswerkopdracht van deze week kijken voor hints ( opdracht 3.2 )

In [2]:
import numpy as np;
from tensorflow.keras.datasets import mnist;
import tensorflow as tf;
import random as rnd;

In [3]:
# 60_000 training images
# 10_000 test images

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [4]:
# Method Block

def OneHotEncodeDigit(label: int):
    """
    One hot encode a single one digit label
    """
    arr = np.zeros((10))
    arr[label] = 1
    return arr

def OneHotEncode(arr):
    """
    One hot encode an array of labels, like the train_images
    """
    return np.array([OneHotEncodeDigit(item) for item in arr])

def normalize_and_vectorize_image(image: np.array):
    """
    Turn a image of variable dimesions (must be square) into a vector and normalize values between 0 and 1
    """
    flat = image.flatten()

    maximum = np.max(flat)
    normalised: np.array = flat / maximum

    return normalised.astype(np.half)


### Labels representatie

Bedenk het volgende voor je mnist dataset:


- In welke range zijn de labels nu (wat is de min/max waarde)?

- Zijn de waardes numeriek of eigenlijk nominaal? Waarom?

- Waarom willen we dit omzetten naar een ander formaat?  

Antwoorden
- De labels zijn 0 t/m 9
- Hoewel de labels nummers vertegenwoordigen zijn de labels nominaal. Want het gaat over afbeeldingen met een beschrijving. Een afbeelding van 1 plus een afbeelding van 2 wordt geen afbeelding van een 3.
- Omdat er geen hieracische structuur is tussen de labels. Alle zijn gelijkwaardig.

We willen de **labels** omzetten naar een formaat die een NN beter kan interpreteren. Gebruik je hiervoor de OneHot Encoder? 

Antwoord
- One hot encoding is uitstekend geschik voor deze vorm van omzettingen omdat deze waarderingsverschil tussen de labels aanbrengt.

In [5]:
test_labels_encoded = OneHotEncode(test_labels)
train_labels_encoded = OneHotEncode(train_labels)

### Split test /training set

Ook hier moet je de test en de training set opspliten. Denk ook na over de representatie van je data. 


Als je pixelwaardes als parameters gebruikt, hoe ga je ze encoden?


Nurale netwerken werken het best wanneer data een vergelijkbare vormen heeft. Daarvoor wordt minmax normalisatie gebruikt.

In [6]:
test_images_normalised = np.array([normalize_and_vectorize_image(image) for image in test_images])
train_images_normalised = np.array([normalize_and_vectorize_image(image) for image in train_images])

### Simple NN: Layers opzet, architectuur

Gebruik keras van tensorflow om een neuraal netwerk op te zetten. De nieuwe manier om **keras** te gebruiken is:

    import tensorflow as tf
    model = tf.keras.Sequential([...])

Let op: voor sommigen zal de oude manier nog werken:


    from tensorflow import keras
    model = keras.Sequential([...])

Voeg de lagen van je neurale netwerk hier toe, dus tussen de blokhaken.
- Voeg de input layer toe. 

        tf.keras.Input(shape=(?,)),

Als parameter (?) kies het aantal input nodes. Hoeveel nodes kies je en waarom?

- Voeg nu een hidden layer toe. Je kunt volgende code gebruiken

        tf.keras.layers.Dense("?", activation=?)

- Kies als eerste parameter ("?") het aantal nodes in deze layer. Hoeveel nodes kies je? Waarom?

- Kies de activation. Wat was activation ook al weer? Woor nu kun je gewoon de ***relu*** gebruiken.

- Je kunt nog meer hidden layers toevoegen. Hoeveel hidden layers voeg je toe? 

- Voeg nu de output layer toe. Code voor de output layer is hetzelfde als hidden, maar met een andere activatie functie. Hoeveel nodes kies je? Kies voor activation="softmax". Wat doet softmax ook al weer?



In [7]:


# Create a simple neural network model

model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(20, activation='relu'),
    tf.keras.layers.Dense(10, activation='sigmoid')
])



### Compileer je model

Gebruik onderstaande code om je model te **compileren**

Wat compile doet: 

- Bepaalt hoe het model leert: adam optimizer
- Bepaalt wat het model probeert te minimaliseren: categorical crossentropy
- Bepaalt wat je wilt zien als prestatie‑indicator: 'accuracy' (bij ?)

Let er op dat je "loss" er niet bij hoeft te zetten omdat die zowieso als metric gebruikt wordt. 

- Wat is accuracy? Wat kun je er verder bij zetten?

De dingen die je hier instelt worden pas later gebruikt, namelijk in de volgende stap - als je 

    .fit 

doet.

In [8]:

# Je kunt het volgende stukje code uitbreiden

model.compile(optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
    ])



### Model trainen

Nu gaan we het model trainen. Bepaal hoeveel epochs je wilt doen en eg hoe groot de batch grootte is.

In [9]:
model.fit(train_images_normalised, train_labels_encoded, epochs=5, batch_size=64, verbose=1)



Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8719 - loss: 0.4782
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9241 - loss: 0.2680
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9346 - loss: 0.2319
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9408 - loss: 0.2070
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9474 - loss: 0.1852


### Maak voorspellingen

Nu kunnen we voorspellingen maken. Gebruik de functie 

    .predict()



In [10]:
index = rnd.randint(0, 10_000)
pred = model.predict(np.array([test_images_normalised[index]]))
print(f"Voorspelde klasse: {pred[0].argmax()} vs {test_labels[index]}")
model.evaluate(test_images_normalised, test_labels_encoded)



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Voorspelde klasse: 2 vs 6
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9475 - loss: 0.1846


[0.18456090986728668, 0.9474999904632568]

### Experimenteer en onderzoek

Experimenteer nu zelf met verschillende settings / aantal hidden nodes / layers / activation functions en andere. Kijk of je algoritme beter en sneller kan voorspellen.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(140, activation='swish'),
    tf.keras.layers.Dense(40, activation='swish'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        "accuracy",
    ])

model.fit(train_images_normalised, train_labels_encoded, epochs=5, batch_size=128, verbose=1)

model.evaluate(test_images_normalised, test_labels_encoded)


Epoch 1/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9030 - loss: 0.3464
Epoch 2/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9571 - loss: 0.1477
Epoch 3/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9699 - loss: 0.1031
Epoch 4/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9766 - loss: 0.0788
Epoch 5/100
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9807 - loss: 0.0631
Epoch 6/100
314/469 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9853 - loss: 0.0491

In [17]:
model.predict(np.array([test_images_normalised[1]]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step


array([[2.0198770e-24, 5.5034764e-22, 1.0000000e+00, 4.0831373e-20,
        0.0000000e+00, 1.7600261e-34, 1.2552196e-22, 3.4620531e-33,
        3.5669188e-22, 0.0000000e+00]], dtype=float32)